In [2]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge, LogisticRegression, Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, classification_report, roc_auc_score
import joblib
import pandas as pd
import numpy as np

In [3]:
# 线性回归直接预测房子价格
"""
    fetch_california_housing()：是函数，不是类；函数调用加括号执行逻辑，返回一个对象
        类比：load_iris()、fetch_20newsgroups()全部都是函数，不是类
        lb：这是函数运行完返回的一个Bunch对象，lb接收这个Bunch实例
        sklearn.utils.Bunch是sklearn自定义的容器类，像字典，但可以用 . 访问属性
"""
lb = fetch_california_housing(data_home='data')

print(f'获取特征值：{lb.data.shape}')
print('--' * 50)
print(lb.data[0])
print(f'目标值：{lb.target}')
print(lb.DESCR)
print('--' * 50)
print(lb.feature_names)

获取特征值：(20640, 8)
----------------------------------------------------------------------------------------------------
[   8.3252       41.            6.98412698    1.02380952  322.
    2.55555556   37.88       -122.23      ]
目标值：[4.526 3.585 3.521 ... 0.923 0.847 0.894]
.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib reposi

In [4]:
# lb.data.shape中样本数也是20640
lb.target.shape

(20640,)

In [5]:
x_train, x_test, y_train, y_test = train_test_split(lb.data, lb.target, test_size=0.25, random_state=1)
print(x_train.shape)
"""
为什么要对特征X进行标准化（梯度类模型：线性回归SGD、神经网络）：
    不同特征单位、量级差别巨大
    举例：
        特征A：房屋面积0～200
        特征B：房间数量1～5
    梯度下降更新权重的时候：
        面积这个特征数值大 -> 损失对它的梯度天然更大，权重更新幅度剧烈
        房间数数值小 -> 梯度很小，更新很慢
    后果：
        损失曲面变成“扁椭圆”，梯度来回震荡，迭代很久才收敛，甚至不收敛
    标准化后所有特征均值为0，方差为1，各个特征梯度在同一个量级，下降稳定，收敛更快
"""
std_x = StandardScaler()
x_train_scaled = std_x.fit_transform(x_train)
x_test_scaled = std_x.transform(x_test)
"""
y_train.reshape(-1, 1):
    y_train 一般是一维数组：(n_samples, )，比如标签向量
    reshape(-1, 1)：把它变成二维：(n_samples, 1)
        -1：自动计算样本数量
        1:代表只有1列（一个特征）
StandardScaler.fit()的输入要求：必须是2D数组【样本数，特征数量】
    一维数组形状(n_samples,)：sklearn会当成样本维度，不是别为【样本x特征】，直接报错
    二维数组(n_samples, 1)：代表n个样本，1个特征(y标签就是这唯一的特征)，符合API规范
    所以需要在fit时进行 y_train.reshape(-1, 1)的操作
"""
std_y = StandardScaler()
temp = y_train.reshape(-1, 1)

y_train_scaled = std_y.fit_transform(y_train.reshape(-1, 1))
print(y_train_scaled.shape)
y_test_scaled = std_y.transform(y_test.reshape(-1, 1))
print(y_test_scaled.shape)

(15480, 8)
(15480, 1)
(5160, 1)


In [6]:
test1 = np.array([1, 2, 3])
print(test1.shape)
print(type(test1))
test1.reshape(-1, 1)

(3,)
<class 'numpy.ndarray'>


array([[1],
       [2],
       [3]])

In [8]:
lr = LinearRegression()

lr.fit(x_train_scaled, y_train_scaled)

# coefficient: n.（数）系数；（物理）率，系数
print('回归系数', lr.coef_)

y_predict_scaled = lr.predict(x_test_scaled)

y_lr_predict = std_y.inverse_transform(y_predict_scaled)

joblib.dump(lr, "./tmp/test.pkl")
print("正规方程测试集里面每个房子的预测价格：", y_predict_scaled[0: 10])
print("inverse_transform里的每个房子的预测价格：", y_lr_predict[0: 10])
print("正规方程的均方误差：", mean_squared_error(y_test_scaled, y_predict_scaled))

回归系数 [[ 0.71942632  0.10518431 -0.23147194  0.26802332 -0.00448136 -0.03495117
  -0.7849086  -0.76307353]]
正规方程测试集里面每个房子的预测价格： [[ 0.039975  ]
 [-0.9856667 ]
 [ 0.54595901]
 [-0.31917221]
 [ 0.65037085]
 [ 1.23359413]
 [ 0.81054876]
 [-0.38917515]
 [-0.28938242]
 [-0.05080248]]
inverse_transform里的每个房子的预测价格： [[2.12391852]
 [0.93825754]
 [2.7088455 ]
 [1.70873764]
 [2.82954754]
 [3.50376456]
 [3.0147162 ]
 [1.62781292]
 [1.74317518]
 [2.01897806]]
正规方程的均方误差： 0.40082431136214186


# 2. 加载保存的模型

In [9]:
model = joblib.load("./tmp/test.pkl")

y_predict_scaled = model.predict(x_test_scaled)

print('保存的模型预测结果：', y_predict_scaled)
print('正规方程均方误差：', mean_squared_error(y_test_scaled, y_predict_scaled))

print('正规方程inverse后的均方误差：', mean_squared_error(std_y.inverse_transform(y_test_scaled), std_y.inverse_transform(y_predict_scaled)))

保存的模型预测结果： [[ 0.039975  ]
 [-0.9856667 ]
 [ 0.54595901]
 ...
 [-0.72237246]
 [ 0.57093569]
 [-0.27655325]]
正规方程均方误差： 0.40082431136214186
正规方程inverse后的均方误差： 0.5356532845422556


In [10]:
y_true = [3, -0.5, 2, 7]
y_pred = [2.5, 0.0, 2, 8]
mean_squared_error(y_true, y_pred)

0.375

In [11]:
# 人工求均方误差
(np.square(3 - 2.5) + np.square(0.5 - 0.0) + 0 + 1) / 4

np.float64(0.375)

# 3 线性回归之梯度下降去进行房价预测

In [12]:
sgd = SGDRegressor(eta0=0.01, penalty='l2', max_iter=1000)

sgd.fit(x_train, y_train)

print('梯度下降的回归系数：', sgd.coef_)

y_predict = sgd.predict(x_test)

print('梯度下降的均方误差：', mean_squared_error(y_test, y_predict))

梯度下降的回归系数： [-8.12223841e+10 -1.27914000e+11 -1.04980177e+11  8.65991409e+10
 -9.07877585e+10 -8.85450749e+11 -1.14499051e+11  5.81510265e+11]
梯度下降的均方误差： 5.538627417315968e+28


In [17]:
w = 1
alpha = 0.1
def loss(w):
    return 2 * w ** 2 + 3 * w + 2
def derivative(w):
    return 4 * w + 3
for i in range(10):
    w = w - alpha * derivative(w)
    print(f'w {w:<20}, 损失{loss(w)}')

w 0.29999999999999993 , 损失3.0799999999999996
w -0.12               , 损失1.6688
w -0.372              , 损失1.160768
w -0.5232             , 损失0.9778764800000002
w -0.61392            , 损失0.9120355328
w -0.6683520000000001 , 损失0.888332791808
w -0.7010112000000001 , 损失0.87979980505088
w -0.72060672         , 损失0.8767279298183168
w -0.732364032        , 损失0.8756220547345941
w -0.7394184192       , 损失0.8752239397044537
